# 01 · Chunking: gdzie przeciąć dokument?

RAG wyszukuje **fragmenty**, nie całe pliki. Przetniemy autorską notatkę naukową Acme na okna słów, akapity, sekcje i fragmenty wyznaczone zmianą tematu. Następnie zadamy to samo pytanie każdemu wariantowi. Ten notebook działa samodzielnie i korzysta wyłącznie z lokalnego modelu embeddingowego.

Po lekcji umiesz wyjaśnić, dlaczego długość, overlap i granica sekcji wpływają na trafność, koszt oraz jakość cytatu. Źródło: `SCI-FORECAST-2011:v1`, fikcyjna notatka metodyczna. Embeddingi są oceną podobieństwa, a nie dowodem prawdziwości.

In [ ]:
# Jedyna komórka z importami i ścieżką projektu.
import re
import sys
from pathlib import Path

import numpy as np

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "models.toml").exists())
sys.path.insert(0, str(ROOT / "notebooks" / "rag"))
from lab import embed_texts, load_document, section_records

## 1. Poznaj dokument

Metadata i wersja pochodzą z pliku, a nie z odpowiedzi modelu. Zwróć uwagę na część o zerowym mianowniku: ma ją później odnaleźć retriever.

In [ ]:
doc = load_document("SCI-FORECAST-2011")
raw_body = doc["body"]
# Pytania ewaluacyjne nie są dowodem: wszystkie strategie dostają ten sam tekst źródłowy.
body = raw_body.split("## Questions for retrieval evaluation", 1)[0].strip()
sections = section_records(("SCI-FORECAST-2011",))
[(row["heading"], len(row["text"].split())) for row in sections]

## 2. Stałe okna, akapity i sekcje

Okno 80 słów daje precyzyjniejszy trafiony fragment, ale łatwo rozcina definicję od zastrzeżenia. Okno 250 słów niesie więcej kontekstu, lecz zużywa więcej miejsca w promptcie. Overlap kopiuje końcówkę poprzedniego fragmentu; zwiększa koszt indeksu i może mnożyć niemal identyczne trafienia. Liczymy tu **słowa** dla czytelności; limit kontekstu modelu dotyczy **tokenów**, więc w rzeczywistym pipeline użyj tokenizera właściwego modelu.

In [ ]:
def word_windows(text, size, overlap=0):
    words = text.split()
    step = size - overlap
    if step <= 0:
        raise ValueError("overlap must be smaller than size")
    return [" ".join(words[start:start + size]) for start in range(0, len(words), step)]


small = word_windows(body, 80, 10)
large = word_windows(body, 250, 30)
no_overlap = word_windows(body, 80, 0)
wide_overlap = word_windows(body, 80, 30)
paragraphs = [part.strip() for part in re.split(r"\n\s*\n", body) if len(part.split()) > 8]
by_section = [row["text"] for row in sections]
[(name, len(parts), round(np.mean([len(p.split()) for p in parts]))) for name, parts in
 [("80 słów", small), ("250 słów", large), ("akapit", paragraphs), ("sekcja", by_section)]]

Pierwsze dwa okna mogą przecinać akapit. Porównaj je z sekcją, która zachowuje nagłówek i granicę tematu. Przy tej samej długości 80 słów większy overlap tworzy więcej fragmentów i powtarza więcej słów w indeksie.

In [ ]:
{"granica_okna": (small[0][-100:], small[1][:100]), "nagłówek_sekcji": sections[1]["heading"],
 "koszt_overlapu": [(name, len(parts), sum(len(part.split()) for part in parts)) for name, parts in
                     [("0 słów", no_overlap), ("10 słów", small), ("30 słów", wide_overlap)]]}

## 3. Chunking semantyczny

Każdy akapit kodujemy osobno. Niski cosinus między sąsiadami sugeruje zmianę tematu. Próg jest **heurystyką zależną od dokumentu**: wykres granic trzeba ocenić ręcznie. Nie zakładamy, że model zna strukturę pliku lub prawdę biznesową.

In [ ]:
vectors = embed_texts(paragraphs)
unit = vectors / np.linalg.norm(vectors, axis=1, keepdims=True)
adjacent = np.sum(unit[:-1] * unit[1:], axis=1)
cutoff = float(np.quantile(adjacent, 0.25))
boundaries = [i + 1 for i, score in enumerate(adjacent) if score < cutoff]
[(i, round(float(adjacent[i - 1]), 3), paragraphs[i][:65]) for i in boundaries]

In [ ]:
semantic = []
start = 0
for stop in [*boundaries, len(paragraphs)]:
    semantic.append("\n\n".join(paragraphs[start:stop]))
    start = stop
[(len(chunk.split()), chunk[:80]) for chunk in semantic]

Granica semantyczna sama nie ogranicza rozmiaru. Jeśli powstanie zbyt długi fragment, dodaj maksymalny limit tokenów i podziel go ponownie, najlepiej na granicy akapitu. Bardzo krótki fragment można dołączyć do sąsiada, zachowując ID sekcji i pochodzenie.

## 4. To samo pytanie, kilka sposobów podziału

Zapytanie i wszystkie fragmenty przechodzą przez **ten sam** model embeddingowy. Wynik to dokładny ranking cosinusowy w małym korpusie; nie myl go z generowaniem odpowiedzi. Kolumna `top1_ma_licznik_i_zero` sprawdza tylko dwa słowa. To nie zastępuje ręcznej oceny, czy fragment wystarcza do odpowiedzi.

Przed podziałem wykluczyliśmy końcową listę **pytań ewaluacyjnych**. Inaczej pytanie mogłoby wygrać ranking, choć nie zawiera odpowiedzi, a porównywane strategie dostałyby różny materiał źródłowy.

In [ ]:
query = "Why can percentage forecast error be undefined for zero demand?"
q = embed_texts([query])[0]
q = q / np.linalg.norm(q)
results = []
for name, chunks in [("80 / overlap 0", no_overlap), ("80 / overlap 10", small),
                     ("80 / overlap 30", wide_overlap), ("250 słów", large),
                     ("sekcja", by_section), ("semantycznie", semantic)]:
    matrix = embed_texts(chunks)
    matrix = matrix / np.linalg.norm(matrix, axis=1, keepdims=True)
    scores = matrix @ q
    best = int(np.argmax(scores))
    results.append({"strategia": name, "fragmentów": len(chunks), "słów_w_top1": len(chunks[best].split()),
                    "cosinus_top1": round(float(scores[best]), 3),
                    "top1_ma_licznik_i_zero": "zero" in chunks[best].lower() and "percent" in chunks[best].lower(),
                    "początek": chunks[best][:100]})
results

Porównaj wynik `cosinus_top1` z kolumną wskazującą obecność dwóch słów. Wysoki cosinus może dotyczyć pokrewnego tematu, a overlap może zarówno uratować zdanie na granicy okna, jak i stworzyć dodatkowe, prawie identyczne trafienia. Dopiero przeczytanie top-1 potwierdzi, czy fragment zawiera pełną odpowiedź.

## 5. Hierarchia parent → child

Indeksujemy krótkie dzieci, ale po trafieniu możemy zwrócić sekcję rodzica. W produkcji należy przechować `child_id`, `parent_id`, wersję i hash tekstu. Rodzic bywa za długi na prompt: można pobrać tylko sąsiednie dzieci albo skrócić sekcję **po** ocenie dowodu.

In [ ]:
children = [(row["id"], i, child) for row in sections for i, child in enumerate(word_windows(row["text"], 80, 10))]
child_vectors = embed_texts([text for _, _, text in children])
child_vectors /= np.linalg.norm(child_vectors, axis=1, keepdims=True)
best_child = children[int(np.argmax(child_vectors @ q))]
parent = next(row for row in sections if row["id"] == best_child[0])
{"child": best_child[0] + f":c{best_child[1]:02d}", "child_words": len(best_child[2].split()),
 "parent_heading": parent["heading"], "parent_words": len(parent["text"].split())}

### Sprawdź samodzielnie

Zmień `size` na 40 i 400 oraz overlap na 0 i 40. Czy top-1 zmienia się? Kiedy przytoczyłbyś dziecko, a kiedy rodzica? W notatkach zapisz także liczbę embeddingów i łączną liczbę słów zwracanych do modelu.